# 📘 Módulo 04: Normalización Completa de Bases de Datos con Diagramas Entidad-Relación (1FN a 5FN & BCNF)
### 🧩 Guía Dócil con Diagramas MER/DER, Tablas 'Antes y Después' y Sentencias SQL Ejecutables en DuckDB
**Asignatura:** Bases de Datos Avanzadas  
**Entorno:** Google Colab / DuckDB (100% In-Memory)  
---
## 🎯 Objetivos de Aprendizaje
1. Visualizar mediante **Diagramas Entidad-Relación (MER/DER)** la evolución de la arquitectura de datos en cada paso de normalización.
2. Comprender la descomposición de esquemas mediante tablas de datos reales 'Antes y Después'.
3. Diagnosticar y resolver las 3 anomalías catastróficas (Inserción, Actualización, Borrado).
4. Aplicar paso a paso: **UNF $\rightarrow$ 1FN $\rightarrow$ 2FN $\rightarrow$ 3FN $\rightarrow$ BCNF $\rightarrow$ 4FN $\rightarrow$ 5FN**.
5. Refactorizar esquemas sucios en **DuckDB** con comandos SQL `CREATE TABLE` e `INSERT INTO ... SELECT DISTINCT`.

In [ ]:
# ==============================================================================
# 🚀 SETUP INICIAL AUTO-CONTENIDO: DUCKDB & GOOGLE COLAB (100% IN-MEMORY)
# ==============================================================================
!pip install duckdb duckdb-engine jupysql -q

import duckdb

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

# Conectar a motor DuckDB en memoria
conn = duckdb.connect(database=':memory:')

%sql conn
print('✅ Entorno DuckDB inicializado para el Módulo 04 (ERD & Normalización).')

## 1. El Estado Inicial: Tabla Plana Sucia (UNF - Unnormalized Form)
Imaginemos que una institución guarda todas sus matrículas y datos en una sola **Tabla Plana Sucia (UNF)**.
### 📐 Diagrama Entidad-Relación Sucio (UNF - Entidad Única Sobrecargada):
```text
  ┌──────────────────────────────────────────────────────────────────┐
  │                    MATRICULAS_UNF (Tabla Plana Sucia)            │
  ├──────────────────────────────────────────────────────────────────┤
  │ estudiante_id (PK) | estudiante_nombre | ciudad | cursos_lista   │
  │ carrera_nombre | facultad_nombre | decano_facultad              │
  └──────────────────────────────────────────────────────────────────┘
```
### 📄 Datos Reales en UNF:
| estudiante_id | estudiante_nombre | ciudad | cursos_lista (No Atómico) | carrera_nombre | facultad_nombre | decano_facultad |
| :---: | :--- | :--- | :--- | :--- | :--- | :--- |
| **101** | Carlos Mendoza | Bogotá | SQL Avanzado, Python AI | Ing. Sistemas | Ingeniería | Dr. Roberto Gómez |
| **102** | Ana María Gómez | Medellín | SQL Avanzado | Ing. Sistemas | Ingeniería | Dr. Roberto Gómez |
| **103** | Luis Fernando Torres | Cali | Redes Cisco, Seguridad Cloud | Redes & Cloud | Ingeniería | Dr. Roberto Gómez |
### 💥 Las 3 Anomalías de esta Tabla Sucia:
1. 🛑 **Inserción:** No podemos crear la carrera `'Inteligencia Artificial'` si no hay estudiantes matriculados.
2. 🔄 **Actualización:** Si el decano `Dr. Roberto Gómez` cambia de nombre, hay que actualizar miles de filas en la tabla.
3. ❌ **Borrado:** Si el estudiante 103 se retira y borramos su fila, **borramos la existencia de la carrera 'Redes & Cloud'**.

In [ ]:
%%sql
-- 0. Tabla Plana Sucia (UNF) en DuckDB
CREATE TABLE IF NOT EXISTS matriculas_unf (
    estudiante_id INT,
    estudiante_nombre VARCHAR,
    ciudad VARCHAR,
    cursos_lista VARCHAR,
    carrera_nombre VARCHAR,
    facultad_nombre VARCHAR,
    decano_facultad VARCHAR
);

INSERT INTO matriculas_unf VALUES
(101, 'Carlos Mendoza', 'Bogotá', 'SQL Avanzado, Python AI', 'Ing. Sistemas', 'Ingeniería', 'Dr. Roberto Gómez'),
(102, 'Ana María Gómez', 'Medellín', 'SQL Avanzado', 'Ing. Sistemas', 'Ingeniería', 'Dr. Roberto Gómez'),
(103, 'Luis Fernando Torres', 'Cali', 'Redes Cisco, Seguridad Cloud', 'Redes & Cloud', 'Ingeniería', 'Dr. Roberto Gómez');

SELECT * FROM matriculas_unf;

## 2. Primera Forma Normal (1FN): Atomicidad & Eliminación de Grupos Repetitivos 🎯
### 📜 Las 3 Reglas Sagradas de la 1FN:
1. **Valores Atómicos:** Cada celda debe contener **un único valor indivisible** (Se prohíben listas como `'SQL Avanzado, Python AI'`).
2. **Sin Columnas Repetidas:** Se prohíben grupos repetitivos horizontales como `curso_1`, `curso_2`, `curso_3`.
3. **Clave Primaria Unívoca:** Cada fila debe ser identificable mediante una Clave Primaria (PK) o Clave Compuesta.
---
### 📐 Diagrama ERD en 1FN (Clave Primaria Compuesta `(estudiante_id, curso_nombre)`):
```text
  ┌──────────────────────────────────────────────────────────────────┐
  │                MATRICULAS_1FN (Fila Única Atómica)               │
  ├──────────────────────────────────────────────────────────────────┤
  │ PK  estudiante_id                                                │
  │ PK  curso_nombre                                                 │
  │     estudiante_nombre | ciudad | carrera_nombre | decano_facultad│
  └──────────────────────────────────────────────────────────────────┘
```
--- 
### 📄 Datos Reales en 1FN (Desglose en Filas Atómicas):
| estudiante_id (PK) | curso_nombre (PK) | estudiante_nombre | ciudad | carrera_nombre | facultad_nombre | decano_facultad |
| :---: | :--- | :--- | :--- | :--- | :--- | :--- |
| **101** | SQL Avanzado | **Carlos Mendoza** | Bogotá | Ing. Sistemas | Ingeniería | Dr. Roberto Gómez |
| **101** | Python AI | **Carlos Mendoza** | Bogotá | Ing. Sistemas | Ingeniería | Dr. Roberto Gómez |
| **102** | SQL Avanzado | Ana María Gómez | Medellín | Ing. Sistemas | Ingeniería | Dr. Roberto Gómez |
| **103** | Redes Cisco | **Luis Fernando Torres** | Cali | Redes & Cloud | Ingeniería | Dr. Roberto Gómez |
| **103** | Seguridad Cloud | **Luis Fernando Torres** | Cali | Redes & Cloud | Ingeniería | Dr. Roberto Gómez |
--- 
### ⚠️ ¡ATENCIÓN DIDÁCTICA! ¿Qué problema NACE en la 1FN?
🔴 **Repetición de Datos en Columna:** Observa cómo al desglosar los cursos en filas individuales, el nombre **'Carlos Mendoza'**, la ciudad **'Bogotá'** y la carrera **'Ing. Sistemas'** se **REPITEN en las columnas** de las filas 101-1 y 101-2.
👉 **Conclusión:** La 1FN logra atomicidad en la celda, **pero genera redundancia de datos repetidos en la columna**. Esta repetición (Dependencia Parcial) es el detonante que nos **OBLIGA a avanzar hacia la 2FN** para separar las entidades.

In [ ]:
%%sql
-- 1. Creación de la Tabla en 1FN (Valores Atómicos)
CREATE TABLE IF NOT EXISTS matriculas_1fn (
    estudiante_id INT,
    curso_nombre VARCHAR, -- Atómico
    estudiante_nombre VARCHAR,
    ciudad VARCHAR,
    carrera_nombre VARCHAR,
    facultad_nombre VARCHAR,
    decano_facultad VARCHAR,
    PRIMARY KEY (estudiante_id, curso_nombre)
);

INSERT INTO matriculas_1fn VALUES
(101, 'SQL Avanzado', 'Carlos Mendoza', 'Bogotá', 'Ing. Sistemas', 'Ingeniería', 'Dr. Roberto Gómez'),
(101, 'Python AI', 'Carlos Mendoza', 'Bogotá', 'Ing. Sistemas', 'Ingeniería', 'Dr. Roberto Gómez'),
(102, 'SQL Avanzado', 'Ana María Gómez', 'Medellín', 'Ing. Sistemas', 'Ingeniería', 'Dr. Roberto Gómez'),
(103, 'Redes Cisco', 'Luis Fernando Torres', 'Cali', 'Redes & Cloud', 'Ingeniería', 'Dr. Roberto Gómez'),
(103, 'Seguridad Cloud', 'Luis Fernando Torres', 'Cali', 'Redes & Cloud', 'Ingeniería', 'Dr. Roberto Gómez');

SELECT * FROM matriculas_1fn;

## 3. Segunda Forma Normal (2FN): Eliminar Dependencias Parciales
### 📐 Diagrama ERD en 2FN (Separación de Entidades 1:N):
```text
  ┌──────────────────┐               ┌──────────────────┐
  │ ESTUDIANTES_2FN  │ 1           N │ INSCRIPCIONES_2FN│
  ├──────────────────┤───────────────├──────────────────┤
  │ PK estudiante_id │               │ PK estudiante_id │
  │    nombre        │               │ PK curso_nombre  │
  │    carrera       │               │    fecha         │
  └──────────────────┘               └──────────────────┘
```
### 🔄 Datos Reales en 2FN (Divididos en 2 Tablas Limpias):
**Tabla 1: `Estudiantes_2FN`**
| estudiante_id (PK) | estudiante_nombre | ciudad | carrera_nombre | facultad_nombre | decano_facultad |
| :---: | :--- | :--- | :--- | :--- | :--- |
| **101** | Carlos Mendoza | Bogotá | Ing. Sistemas | Ingeniería | Dr. Roberto Gómez |
| **102** | Ana María Gómez | Medellín | Ing. Sistemas | Ingeniería | Dr. Roberto Gómez |
| **103** | Luis Fernando Torres | Cali | Redes & Cloud | Ingeniería | Dr. Roberto Gómez |
**Tabla 2: `Inscripciones_2FN`**
| estudiante_id (FK) | curso_nombre (PK) | fecha_inscripcion |
| :---: | :--- | :---: |
| **101** | SQL Avanzado | 2024-01-15 |
| **101** | Python AI | 2024-01-16 |
| **102** | SQL Avanzado | 2024-01-15 |
| **103** | Redes Cisco | 2024-01-20 |

In [ ]:
%%sql
-- 2. Creación de Tablas en 2FN en DuckDB
CREATE TABLE IF NOT EXISTS estudiantes_2fn (
    estudiante_id INT PRIMARY KEY,
    estudiante_nombre VARCHAR,
    ciudad VARCHAR,
    carrera_nombre VARCHAR,
    facultad_nombre VARCHAR,
    decano_facultad VARCHAR
);

CREATE TABLE IF NOT EXISTS inscripciones_2fn (
    estudiante_id INT REFERENCES estudiantes_2fn(estudiante_id),
    curso_nombre VARCHAR,
    PRIMARY KEY (estudiante_id, curso_nombre)
);

INSERT INTO estudiantes_2fn 
SELECT DISTINCT estudiante_id, estudiante_nombre, ciudad, carrera_nombre, facultad_nombre, decano_facultad
FROM matriculas_1fn;

INSERT INTO inscripciones_2fn 
SELECT estudiante_id, curso_nombre 
FROM matriculas_1fn;

SELECT * FROM estudiantes_2fn;

## 4. Tercera Forma Normal (3FN): Eliminar Dependencias Transitivas
### 📐 Diagrama ERD en 3FN (Descomposición Total en Entidades Independientes):
```text
  ┌──────────────┐ 1    N ┌──────────────┐ 1    N ┌──────────────┐ 1    N ┌──────────────┐
  │ FACULTADES   │────────│ CARRERAS     │────────│ ESTUDIANTES  │────────│ INSCRIPCION  │
  ├──────────────┤        ├──────────────┤        ├──────────────┤        ├──────────────┤
  │ PK facultad  │        │ PK carrera   │        │ PK estudiante│        │ PK estudiante│
  │    decano    │        │ FK facultad  │        │ FK carrera   │        │ PK curso     │
  └──────────────┘        └──────────────┘        └──────────────┘        └──────────────┘
```
### 🔄 Datos Reales en 3FN (Sin Dependencias Transitivas $A \rightarrow B \rightarrow C$):
**Tabla 1: `Facultades_3FN`**
| facultad_id (PK) | facultad_nombre | decano_facultad |
| :---: | :--- | :--- |
| **1** | Ingeniería | Dr. Roberto Gómez |
| **2** | Ciencias Económicas | Dra. Elena Restrepo |
**Tabla 2: `Carreras_3FN`**
| carrera_id (PK) | carrera_nombre | facultad_id (FK) |
| :---: | :--- | :---: |
| **10** | Ing. Sistemas | 1 |
| **20** | Redes & Cloud | 1 |
**Tabla 3: `Estudiantes_3FN`**
| estudiante_id (PK) | estudiante_nombre | ciudad | carrera_id (FK) |
| :---: | :--- | :--- | :---: |
| **101** | Carlos Mendoza | Bogotá | 10 |
| **102** | Ana María Gómez | Medellín | 10 |
| **103** | Luis Fernando Torres | Cali | 20 |

In [ ]:
%%sql
-- 3. Creación del Esquema Normalizado en 3FN en DuckDB
CREATE TABLE IF NOT EXISTS facultades_3fn (
    facultad_id INT PRIMARY KEY,
    facultad_nombre VARCHAR NOT NULL,
    decano_facultad VARCHAR
);

CREATE TABLE IF NOT EXISTS carreras_3fn (
    carrera_id INT PRIMARY KEY,
    carrera_nombre VARCHAR NOT NULL,
    facultad_id INT REFERENCES facultades_3fn(facultad_id)
);

CREATE TABLE IF NOT EXISTS estudiantes_3fn (
    estudiante_id INT PRIMARY KEY,
    estudiante_nombre VARCHAR NOT NULL,
    ciudad VARCHAR,
    carrera_id INT REFERENCES carreras_3fn(carrera_id)
);

INSERT INTO facultades_3fn VALUES (1, 'Ingeniería', 'Dr. Roberto Gómez');
INSERT INTO carreras_3fn VALUES (10, 'Ing. Sistemas', 1), (20, 'Redes & Cloud', 1);
INSERT INTO estudiantes_3fn VALUES 
(101, 'Carlos Mendoza', 'Bogotá', 10),
(102, 'Ana María Gómez', 'Medellín', 10),
(103, 'Luis Fernando Torres', 'Cali', 20);

-- Reconstrucción perfecta con JOIN sin redundancia:
SELECT 
    e.estudiante_id,
    e.estudiante_nombre,
    e.ciudad,
    c.carrera_nombre,
    f.facultad_nombre,
    f.decano_facultad
FROM estudiantes_3fn e
JOIN carreras_3fn c ON e.carrera_id = c.carrera_id
JOIN facultades_3fn f ON c.facultad_id = f.facultad_id;

## 5. Cuarta Forma Normal (4FN): Eliminar Dependencias Multivaluadas Independientes 💥
### 💡 ¿Por qué necesitamos 4FN si ya estamos en 3FN?
Una tabla puede estar en **3FN** y seguir teniendo un problema severo de **redondancia por Dependencia Multivaluada ($A \twoheadrightarrow B$)**.
Esto ocurre cuando una entidad tiene **dos o más relaciones multivaluadas independientes entre sí** (por ejemplo: las *habilidades* de un estudiante y los *idiomas* que habla).
---
### 📄 Ejemplo Práctico: Tabla 3FN con Explosión Combinatoria (Violación de 4FN)
Si guardamos ambas habilidades e idiomas en una sola tabla `(estudiante_id, habilidad, idioma)`:
| estudiante_id | habilidad | idioma |
| :---: | :--- | :--- |
| **101** | SQL | Español |
| **101** | SQL | Inglés |
| **101** | Python | Español |
| **101** | Python | Inglés |
💥 **El Problema:** Como la habilidad y el idioma no están relacionados entre sí, si el estudiante aprende una nueva habilidad (`Java`), nos vemos obligados a insertar **2 filas nuevas** (una para Español y otra para Inglés). ¡Esto produce una **explosión combinatoria de datos**!
---
### 📐 Diagrama en 4FN (Descomposición en 2 Tablas Independientes 1:N):
```text
  ┌───────────────────────┐             ┌───────────────────────┐
  │ ESTUDIANTE_HABILIDADES│             │   ESTUDIANTE_IDIOMAS  │
  ├───────────────────────┤             ├───────────────────────┤
  │ PK,FK estudiante_id   │             │ PK,FK estudiante_id   │
  │ PK    habilidad       │             │ PK    idioma          │
  └───────────────────────┘             └───────────────────────┘
```
### ✅ Tablas Normalizadas en 4FN:
**Tabla 1: `estudiante_habilidades_4fn` (Solo 2 filas)**
| estudiante_id | habilidad |
| :---: | :--- |
| 101 | SQL |
| 101 | Python |
**Tabla 2: `estudiante_idiomas_4fn` (Solo 2 filas)**
| estudiante_id | idioma |
| :---: | :--- |
| 101 | Español |
| 101 | Inglés |

In [ ]:
%%sql
-- 5. Demostración de 4FN en DuckDB

-- Crear Tablas Independientes en 4FN (Sin Producto Cartesiano Redundante)
CREATE TABLE IF NOT EXISTS estudiante_habilidades_4fn (
    estudiante_id INT,
    habilidad VARCHAR,
    PRIMARY KEY (estudiante_id, habilidad)
);

CREATE TABLE IF NOT EXISTS estudiante_idiomas_4fn (
    estudiante_id INT,
    idioma VARCHAR,
    PRIMARY KEY (estudiante_id, idioma)
);

-- Insertar Datos sin duplicación innecesaria
INSERT INTO estudiante_habilidades_4fn VALUES (101, 'SQL'), (101, 'Python');
INSERT INTO estudiante_idiomas_4fn VALUES (101, 'Español'), (101, 'Inglés');

-- Consulta limpia de Habilidades
SELECT * FROM estudiante_habilidades_4fn WHERE estudiante_id = 101;

## 6. Retos Prácticos de Normalización para el Estudiante 🏋️‍♂️
### ✏️ Enunciado del Reto:
Dada la tabla sucia de facturación en UNF:
`facturas_unf (factura_id, fecha, cliente_nombre, ciudad_cliente, vendedor_nombre, depto_vendedor, producto_nombre, cantidad, precio)`
1. **Reto 1:** Diseñar la estructura relacional normalizada en **3FN** (Entidades, PK y FKs).
2. **Reto 2:** Escribir el script DDL `CREATE TABLE` en SQL (DuckDB) para implementar el esquema 3FN.
3. **Reto 3:** Insertar datos de prueba normalizados e intentar una consulta con `JOIN` que reconstruya la factura completa.

In [1]:
-- 🛠 Solución Oficial del Reto - Módulo 04: Normalización 3FN

-- 1. Tabla de Clientes (Elimina la dependencia transitiva de ciudad/nombre)
CREATE TABLE IF NOT EXISTS clientes_reto (
    cliente_id INT PRIMARY KEY,
    cliente_nombre VARCHAR NOT NULL,
    ciudad_cliente VARCHAR NOT NULL
);

-- 2. Tabla de Vendedores (Elimina la dependencia transitiva del departamento)
CREATE TABLE IF NOT EXISTS vendedores_reto (
    vendedor_id INT PRIMARY KEY,
    vendedor_nombre VARCHAR NOT NULL,
    depto_vendedor VARCHAR NOT NULL
);

-- 3. Tabla de Productos (Atómica)
CREATE TABLE IF NOT EXISTS productos_reto (
    producto_id INT PRIMARY KEY,
    producto_nombre VARCHAR NOT NULL,
    precio_base DECIMAL(12,2) NOT NULL
);

-- 4. Tabla de Facturas Encabezado (1:N con Clientes y Vendedores)
CREATE TABLE IF NOT EXISTS facturas_reto (
    factura_id INT PRIMARY KEY,
    fecha DATE NOT NULL,
    cliente_id INT REFERENCES clientes_reto(cliente_id),
    vendedor_id INT REFERENCES vendedores_reto(vendedor_id)
);

-- 5. Tabla de Detalle Factura (Relación N:M entre Factura y Producto)
CREATE TABLE IF NOT EXISTS detalle_factura_reto (
    factura_id INT REFERENCES facturas_reto(factura_id),
    producto_id INT REFERENCES productos_reto(producto_id),
    cantidad INT CHECK (cantidad > 0),
    precio_unitario DECIMAL(12,2) NOT NULL,
    PRIMARY KEY (factura_id, producto_id)
);

-- 📥 Poblar Datos de Prueba Normalizados:
INSERT INTO clientes_reto VALUES (1, 'Carlos Mendoza', 'Bogotá'), (2, 'Ana Gómez', 'Medellín');
INSERT INTO vendedores_reto VALUES (10, 'Laura Patiño', 'Ventas Corporativas'), (20, 'Andrés Ruiz', 'Retail');
INSERT INTO productos_reto VALUES (101, 'Laptop Dell XPS', 4500000.00), (102, 'Mouse Inalámbrico', 85000.00);

INSERT INTO facturas_reto VALUES (5001, '2026-10-05', 1, 10), (5002, '2026-10-06', 2, 20);
INSERT INTO detalle_factura_reto VALUES (5001, 101, 1, 4500000.00), (5001, 102, 2, 85000.00), (5002, 102, 1, 85000.00);

-- 🔍 Consulta de Reconstrucción de Factura con JOINs 3FN:
SELECT 
    f.factura_id,
    f.fecha,
    c.cliente_nombre,
    c.ciudad_cliente,
    v.vendedor_nombre,
    p.producto_nombre,
    df.cantidad,
    df.precio_unitario,
    (df.cantidad * df.precio_unitario) AS subtotal
FROM facturas_reto f
JOIN clientes_reto c ON f.cliente_id = c.cliente_id
JOIN vendedores_reto v ON f.vendedor_id = v.vendedor_id
JOIN detalle_factura_reto df ON f.factura_id = df.factura_id
JOIN productos_reto p ON df.producto_id = p.producto_id;


## 7. Explicación Complementaria: Quinta Forma Normal (5FN / PJNF) 🎓
### 💡 ¿Qué es la Quinta Forma Normal (5FN / Project-Join Normal Form)?
La **5FN** aborda casos avanzados donde existen **Dependencias de Proyección-Unión (Join Dependencies - JD)** en relaciones de 3 o más entidades (relaciones ternarias).
Una tabla está en **5FN** si no puede descomponerse en tablas más pequeñas que luego, al ser unidas con `JOIN`, generen **filas fantasma/espurias** (datos falsos que nunca existieron).
---
### 📄 Ejemplo Práctico de Negocio: Asignación Ternaria Cíclica
Imaginemos una regla comercial de Cartera Corporativa entre 3 entidades:
1. **Vendedor** ($V$)
2. **Cliente Corporativo** ($C$)
3. **Marca de Producto** ($M$)
**Regla de Negocio:** La venta $(V, C, M)$ es válida **ÚNICAMENTE SI**:
- El vendedor $V$ está certificado en la marca $M$.
- El cliente $C$ compra la marca $M$.
- El vendedor $V$ atiende a la cuenta del cliente $C$.
---
### 📐 Esquema Descompuesto en 5FN (3 Proyecciones Binarias Simétricas):
```text
  ┌───────────────────────┐   ┌───────────────────────┐   ┌───────────────────────┐
  │   VENDEDOR_MARCAS     │   │    CLIENTE_MARCAS     │   │   VENDEDOR_CLIENTES   │
  ├───────────────────────┤   ├───────────────────────┤   ├───────────────────────┤
  │ PK,FK vendedor_id     │   │ PK,FK cliente_id      │   │ PK,FK vendedor_id     │
  │ PK    marca           │   │ PK    marca           │   │ PK,FK cliente_id      │
  └───────────────────────┘   └───────────────────────┘   └───────────────────────┘
```
---

In [ ]:
%%sql
-- 7. Demostración Complementaria de 5FN en DuckDB

-- 1. Proyección 1: Vendedor <-> Marca
CREATE TABLE IF NOT EXISTS vendedor_marcas_5fn (
    vendedor_id INT,
    marca VARCHAR,
    PRIMARY KEY (vendedor_id, marca)
);

-- 2. Proyección 2: Cliente <-> Marca
CREATE TABLE IF NOT EXISTS cliente_marcas_5fn (
    cliente_id INT,
    marca VARCHAR,
    PRIMARY KEY (cliente_id, marca)
);

-- 3. Proyección 3: Vendedor <-> Cliente
CREATE TABLE IF NOT EXISTS vendedor_clientes_5fn (
    vendedor_id INT,
    cliente_id INT,
    PRIMARY KEY (vendedor_id, cliente_id)
);

-- Poblar Proyecciones en 5FN
INSERT INTO vendedor_marcas_5fn VALUES (10, 'Dell'), (10, 'HP');
INSERT INTO cliente_marcas_5fn VALUES (1, 'Dell'), (1, 'HP');
INSERT INTO vendedor_clientes_5fn VALUES (10, 1);

-- 🔍 Reconstrucción 5FN sin filas fantasma (JOIN Ternario de las 3 Proyecciones):
SELECT 
    vc.vendedor_id,
    vc.cliente_id,
    vm.marca
FROM vendedor_clientes_5fn vc
JOIN vendedor_marcas_5fn vm ON vc.vendedor_id = vm.vendedor_id
JOIN cliente_marcas_5fn cm ON vc.cliente_id = cm.cliente_id AND vm.marca = cm.marca;